# Range Forecaster

Where a price is likely to be 5 to 90 days from now: a **wide range** meant to hold the real price 80% of the time and a **tight range** meant to hold it 50%. It gives no up-or-down call, because nothing tested could make one reliably.

**How it was tested.** Every setting was picked on 2000 to 2018 and then checked once on 2019 to today, on 471 S&P 500 stocks, 18 indexes and sector funds, 26 commodity futures and 3 cryptocurrencies. On those newer years the wide range held the price 78% to 81% of the time.

**How to use**
1. Click **Runtime → Run all**. The first run takes about a minute.
2. Change the ticker in **Step 2**, then press the play buttons on Steps 2, 3 and 4.

Tickers that work: stocks `AAPL`, `NVDA`; indexes `^GSPC`; funds `SPY`, `QQQ`; crypto `BTC-USD`, `ETH-USD`; commodity futures `GC=F` (gold), `CL=F` (crude oil), `ZC=F` (corn), `SI=F` (silver).

This is a measured range, not advice to buy or sell.

In [ ]:
#@title 1) Setup  (run once per session, about 1 minute)
import os, subprocess, sys
REPO = "https://github.com/danieljkoo92/ui-ux-pro-max-mine.git"
HOME = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(HOME)
if not os.path.isdir("forecaster"):
    ok = subprocess.run(f"git clone -q --depth 1 -b forecaster-v2 {REPO} forecaster", shell=True).returncode == 0
    if not ok:
        subprocess.run(f"git clone -q --depth 1 {REPO} forecaster", shell=True, check=True)
os.chdir(os.path.join(HOME, "forecaster", "tools", "kronos-forecaster"))
subprocess.run(f"{sys.executable} -m pip install -q -U yfinance", shell=True, check=True)
print("Downloading the options market's volatility readings (VIX and others)...")
r = subprocess.run([sys.executable, "vol_data.py"], capture_output=True, text=True)
print("\n".join(l for l in r.stdout.splitlines() if l.strip()))
print("Ready.")


In [ ]:
#@title 2) Settings  (change these, then run 3 and 4)
TICKER = "BTC-USD"  #@param {type:"string"}
DAYS_AHEAD = 90  #@param {type:"slider", min:5, max:90, step:5}
PRICE_LEVELS = "57747.77, 100000"  #@param {type:"string"}
t = TICKER.strip().upper()
safe = t.replace("^", "_").replace("=", "_")
levels = [x.strip() for x in PRICE_LEVELS.split(",") if x.strip()]
print(f"{t}  |  {DAYS_AHEAD} days ahead  |  price levels to check: {', '.join(levels) or 'none'}")


In [ ]:
#@title 3) The forecast
import subprocess, sys
from IPython.display import Image, display
cmd = [sys.executable, "forecast.py", t, str(DAYS_AHEAD)] + (["--level"] + levels if levels else [])
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout.split("Saved to")[0])
if r.returncode != 0:
    print("Something went wrong:\n" + r.stderr[-1500:])
else:
    display(Image(f"out/{safe}_{DAYS_AHEAD}d_range/forecast.png"))


In [ ]:
#@title 4) What it would have said in past years
import json, re
r = subprocess.run([sys.executable, "history_view.py", t, str(DAYS_AHEAD)], capture_output=True, text=True)
if r.returncode != 0:
    print("Something went wrong:\n" + r.stderr[-1500:])
else:
    s = json.loads(r.stdout[:r.stdout.index("}") + 1])
    print(f"{s['name']}: every {DAYS_AHEAD} days since the data starts, the range the forecaster would have drawn that day.")
    print(f"  Price ended inside the wide range {s['fans_held']} of {s['fans_drawn']} times.")
    print(f"  Checked every 10 days: held {s['held_every_10_days']:.0%} overall, {s['held_2019_on']:.0%} since 2019.")
    print(f"  Worst year: {s['worst_year']}, when it held {s['worst_year_held']:.0%}.")
    display(Image(f"out/history/{safe}_{DAYS_AHEAD}d_history.png"))


In [ ]:
#@title 5) Optional: a whole watchlist
WATCHLIST = "^GSPC, AAPL, NVDA, GC=F, CL=F, BTC-USD"  #@param {type:"string"}
import pandas as pd
rows = []
for tk in [x.strip().upper() for x in WATCHLIST.split(",") if x.strip()]:
    r = subprocess.run([sys.executable, "forecast.py", tk, str(DAYS_AHEAD)], capture_output=True, text=True)
    line = next((l for l in r.stdout.splitlines() if l.startswith("Wide range")), "")
    m = re.search(r"\(([-+][\d.]+%) to ([-+][\d.]+%)\)", line)
    held = re.search(r"has held (\d+%)", line)
    rows.append({"ticker": tk, "wide range low": m.group(1) if m else "error", "wide range high": m.group(2) if m else "",
                 "has held": held.group(1) if held else ""})
table = pd.DataFrame(rows)
try:
    display(table)
except NameError:
    print(table.to_string(index=False))


## How to read it

- **Wide range (light blue):** where the price should end up 80% of the time. "Has held" is how often it really did on 2019 to today, with each forecast made from the data known that day.
- **Tight range (darker blue):** where it should end up half the time.
- **Middle of the range:** not a prediction of direction. Stocks drift up over time, so the middle usually sits a little above today's price.
- **Chance it touches a level:** a price can dip to a level during the window and recover. Touching is roughly twice as likely as ending past it.
- **When it missed:** how far past the edge the price landed, the times it did.

## What goes into the range

1. **How much the price really moves,** from the last 60 and 250 days (each day's open, high, low and close for stocks).
2. **The VIX** (the options market's forecast of S&P 500 swings) scales stock and fund ranges: narrower when markets are calm, wider when they are nervous.
3. **Re-sized every month** so the last few years held 80%.
4. **Each ticker's own record** nudges its range, leaning on its group when history is thin.
5. **Earnings** (30-day stock ranges): narrower when no report is due, wider when one is.

## Honest limits

- **Sudden crashes break it.** In 2008 the 90-day range held only about 60% of the time.
- **Runaway trends break it on the upside.** Gold since 2019 and Nvidia held about 70% to 75%.
- **No direction call.** 99 chart rules from trading books and videos were tested, and none beat luck on both the older and newer years.
- **Futures prices are the front contract.** When it expires, the price series jumps to the next contract.

The earlier Kronos notebook is still in this folder (`Kronos_Forecaster.ipynb`). On daily charts Kronos called direction at coin-flip accuracy and its "80% range" held only 9% to 22% of the time, which is why this notebook replaced it.